# Run **Expedited Internet Bypass Protocol** on FABRIC Nodes

## Input Required Information

| Variable | Use |
| --- | --- |
| SLICE_NAME    | Name of the slice you wish to work on. |
| NODE_TO_FAIL | Node which will lose access to an interface. |
| INTF_TO_FAIL | Interface that will be failed on the node. |

In [43]:
SLICE_NAME = "Eibp_PTPLARGE132"

#change based on slice details
NODE_TO_FAIL = "C1"
INTF_TO_FAIL = "eth3"

## Access the Slice

The orchestrator class is initalized, which also means the slice and its nodes are now accessable as well.

In [44]:
from FabUtils import FabOrchestrator

try:
    manager = FabOrchestrator(SLICE_NAME)
    
except Exception as e:
    print(f"Exception: {e}")

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

try: 
    fablib = fablib_manager()                
    fablib.show_config()
except Exception as e:
    print(f"Exception: {e}")   

slice = fablib.get_slice(name=SLICE_NAME)    
for node in slice.get_nodes():# nirmala addded this to avoid loosing access to slice nodes. 
    node.execute("sudo systemctl stop NetworkManager.service")

Slice name: Eibp_PTPLARGE132
Slice and nodes were acquired successfully.


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,787adfc9-d37e-42f2-8efe-8e32793e0bb8
Bastion Host,bastion.fabric-testbed.net
Bastion Username,tm3886_0000190412
Bastion Private Key File,/home/fabric/work/fabric_config/Fabric_Bastion_Key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key
Sites to avoid,


## Delete the Log from a Prior Test if Necessary

In [45]:
rmLogCmd = "rm EIBP_*.log"
manager.executeCommandsParallel(rmLogCmd, prefixList="A,D,C")

Starting command on node C1
Command to execute: rm EIBP_*.log
Starting command on node C2
Command to execute: rm EIBP_*.log
Starting command on node C3
Command to execute: rm EIBP_*.log
Starting command on node D1
Command to execute: rm EIBP_*.log
Starting command on node D2
Command to execute: rm EIBP_*.log
Starting command on node D3
Command to execute: rm EIBP_*.log
Starting command on node D4
Command to execute: rm EIBP_*.log
Starting command on node D5
Command to execute: rm EIBP_*.log
Starting command on node A1
Command to execute: rm EIBP_*.log
Starting command on node A2
Command to execute: rm EIBP_*.log
Starting command on node A3
Command to execute: rm EIBP_*.log
Starting command on node A4
Command to execute: rm EIBP_*.log
Starting command on node A5
Command to execute: rm EIBP_*.log

==== C1 RESULTS ====
stdout:

stderr:


==== C2 RESULTS ====
stdout:

stderr:


==== C3 RESULTS ====
stdout:

stderr:


==== D1 RESULTS ====
stdout:

stderr:


==== D2 RESULTS ====
stdout:

std

## EIBP **Initial Convergence**

Delay by a bit to get everything working first. The spines are started first, then 5 seconds later the leaves are.

In [46]:
import time

with open('config_large_13node.txt','r') as file:
    config_data = {}
    
    for line in file:
        parts = line.strip().split(',')
        if len(parts) == 2:
            node_name, command = parts
            config_data[node_name] = command
            

    for node_name, command in config_data.items():
        print(f"Node: {node_name}, Command: {command}")
        startCmd = f"tmux new-session -d -s MNLR 'cd ~/SRC_Code; {command}'"
        if node_name in ("A1"):
            print("Waiting 30 seconds")
            time.sleep(30)
        print(startCmd)
        manager.executeCommandsParallel(startCmd, prefixList=node_name)
        time.sleep(5)

Node: C1, Command: sudo ./MNLR -T 1 -L 1.1 -N 1
tmux new-session -d -s MNLR 'cd ~/SRC_Code; sudo ./MNLR -T 1 -L 1.1 -N 1'
Starting command on node C1
Command to execute: tmux new-session -d -s MNLR 'cd ~/SRC_Code; sudo ./MNLR -T 1 -L 1.1 -N 1'

==== C1 RESULTS ====
stdout:

stderr:

Node: C2, Command: sudo ./MNLR -T 1 -L 1.2 -N 1
tmux new-session -d -s MNLR 'cd ~/SRC_Code; sudo ./MNLR -T 1 -L 1.2 -N 1'
Starting command on node C2
Command to execute: tmux new-session -d -s MNLR 'cd ~/SRC_Code; sudo ./MNLR -T 1 -L 1.2 -N 1'

==== C2 RESULTS ====
stdout:

stderr:

Node: C3, Command: sudo ./MNLR -T 1 -L 1.3 -N 1
tmux new-session -d -s MNLR 'cd ~/SRC_Code; sudo ./MNLR -T 1 -L 1.3 -N 1'
Starting command on node C3
Command to execute: tmux new-session -d -s MNLR 'cd ~/SRC_Code; sudo ./MNLR -T 1 -L 1.3 -N 1'

==== C3 RESULTS ====
stdout:

stderr:

Node: D1, Command: sudo ./MNLR -T 2 -N 1
tmux new-session -d -s MNLR 'cd ~/SRC_Code; sudo ./MNLR -T 2 -N 1'
Starting command on node D1
Command to e

In [47]:
import time
time.sleep(75) # the code stabilizes after 75 seconds and then onyl we should delete

# wait for a minute 

# EIBP **Reconvergence** Testing (Optional/Only run it when needed otherwise do this process manually)

## Take the Interface down

This code brings down a network interface specified by {INTF_TO_FAIL} across multiple nodes and then retrieves IP addresses from the specified nodes in parallel using Python's manager.executeCommandsParallel function.

In [48]:
# Take the specified interface down
#intfName = f"{NODE_TO_FAIL}-{INTF_TO_FAIL}-p1"
#intf = manager.slice.get_interface(intfName)
#intf.ip_link_down()

downCmd = f"sudo ip link set down {INTF_TO_FAIL}"
manager.executeCommandsParallel(downCmd, prefixList=NODE_TO_FAIL)
downCmd = f"sudo ip addr"
manager.executeCommandsParallel(downCmd, prefixList=NODE_TO_FAIL)

Starting command on node C1
Command to execute: sudo ip link set down eth3

==== C1 RESULTS ====
stdout:

stderr:

Starting command on node C1
Command to execute: sudo ip addr

==== C1 RESULTS ====
stdout:
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9000 qdisc fq_codel state UP group default qlen 1000
    link/ether fa:16:3e:e1:07:9c brd ff:ff:ff:ff:ff:ff
    altname enp3s0
    inet 10.20.4.190/23 brd 10.20.5.255 scope global dynamic noprefixroute eth0
       valid_lft 86002sec preferred_lft 86002sec
    inet6 fe80::f816:3eff:fee1:79c/64 scope link 
       valid_lft forever preferred_lft forever
3: eth1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default q

In [49]:
time.sleep(40)

# wait for a minute - record time before KILL 

In [50]:
stopCmd = "date +%s.%6N > stop_time.txt"
manager.executeCommandsParallel(stopCmd, prefixList=NODE_TO_FAIL)

Starting command on node C1
Command to execute: date +%s.%6N > stop_time.txt

==== C1 RESULTS ====
stdout:

stderr:



In [51]:
time.sleep(10)

## Stop EIBP on Each Node
# let us kill from a1, a2, d1, d2 and c1

In [52]:
stopCmd = "tmux kill-session -t MNLR"
manager.executeCommandsParallel(stopCmd, prefixList="A")
manager.executeCommandsParallel(stopCmd, prefixList="D")
manager.executeCommandsParallel(stopCmd, prefixList="C")

Starting command on node A1
Command to execute: tmux kill-session -t MNLR
Starting command on node A2
Command to execute: tmux kill-session -t MNLR
Starting command on node A3
Command to execute: tmux kill-session -t MNLR
Starting command on node A4
Command to execute: tmux kill-session -t MNLR
Starting command on node A5
Command to execute: tmux kill-session -t MNLR

==== A1 RESULTS ====
stdout:

stderr:


==== A2 RESULTS ====
stdout:

stderr:


==== A3 RESULTS ====
stdout:

stderr:


==== A4 RESULTS ====
stdout:

stderr:


==== A5 RESULTS ====
stdout:

stderr:

Starting command on node D1
Command to execute: tmux kill-session -t MNLR
Starting command on node D2
Command to execute: tmux kill-session -t MNLR
Starting command on node D3
Command to execute: tmux kill-session -t MNLR
Starting command on node D4
Command to execute: tmux kill-session -t MNLR
Starting command on node D5
Command to execute: tmux kill-session -t MNLR

==== D1 RESULTS ====
stdout:

stderr:


==== D2 RESULTS ===

In [53]:
time.sleep(10)

## Collect Log Results

Now that the nodes have logged updates to their respective log files, they need to be downloaded to be analyzed.

In [54]:
import os

LOG_DIR_PATH = "logs_large_13node"
LOG_NAME = "EIBP_{name}.log"
logPath = os.path.join(LOG_DIR_PATH, LOG_NAME)

# If the logs directory does not already exist, create it
if not os.path.exists(LOG_DIR_PATH):
    os.makedirs(LOG_DIR_PATH)
    
manager.downloadFilesParallel(logPath, LOG_NAME, prefixList="C,D,A", addNodeName=True)


Starting download on node C1
File to download: EIBP_C1.log
Location of download: logs_large_13node/EIBP_C1.log
Starting download on node C2
File to download: EIBP_C2.log
Location of download: logs_large_13node/EIBP_C2.log
Starting download on node C3
File to download: EIBP_C3.log
Location of download: logs_large_13node/EIBP_C3.log
Starting download on node D1
File to download: EIBP_D1.log
Location of download: logs_large_13node/EIBP_D1.log
Starting download on node D2
File to download: EIBP_D2.log
Location of download: logs_large_13node/EIBP_D2.log
Starting download on node D3
File to download: EIBP_D3.log
Location of download: logs_large_13node/EIBP_D3.log
Starting download on node D4
File to download: EIBP_D4.log
Location of download: logs_large_13node/EIBP_D4.log
Starting download on node D5
File to download: EIBP_D5.log
Location of download: logs_large_13node/EIBP_D5.log
Starting download on node A1
File to download: EIBP_A1.log
Location of download: logs_large_13node/EIBP_A1.log
S

In [55]:
#nirmala added this to restart network manager, else we loose access to the nodes in the slice
from FabUtils import FabOrchestrator
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
try: 
    
    fablib = fablib_manager()
except Exception as e:
    print(f"Exception: {e}")

slice = fablib.get_slice(name=SLICE_NAME)
for node in slice.get_nodes():# nirmala addded this to avoid loosing access to slice nodes. 
    node.execute("sudo systemctl start NetworkManager.service")

## Bring the Interface Back Up

In [56]:
#intf.ip_link_up()
upCmd = f"sudo ip link set up {INTF_TO_FAIL}"
manager.executeCommandsParallel(upCmd, prefixList=NODE_TO_FAIL)
upCmd = f"sudo ip addr"
manager.executeCommandsParallel(upCmd, prefixList=NODE_TO_FAIL)

Starting command on node C1
Command to execute: sudo ip link set up eth3

==== C1 RESULTS ====
stdout:

stderr:

Starting command on node C1
Command to execute: sudo ip addr

==== C1 RESULTS ====
stdout:
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9000 qdisc fq_codel state UP group default qlen 1000
    link/ether fa:16:3e:e1:07:9c brd ff:ff:ff:ff:ff:ff
    altname enp3s0
    inet 10.20.4.190/23 brd 10.20.5.255 scope global dynamic noprefixroute eth0
       valid_lft 86368sec preferred_lft 86368sec
    inet6 fe80::f816:3eff:fee1:79c/64 scope link 
       valid_lft forever preferred_lft forever
3: eth1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qle